# 04 · Reversible, pushed to the maximum batch

First the ceiling, for both stacks: each candidate batch trains 4 steps in a fresh process, with the MPS
allocator capped at the recommended working set (so out-of-memory is a clean failure, not swap). The
search doubles until a batch fails and then bisects.

Then the reversible model trains the same 50M tokens at its own ceiling. A bigger batch means fewer
optimizer steps for the same tokens, so the peak learning rate is scaled by sqrt(batch / 32) (the usual
square-root rule for Adam), with 5% warmup. Everything else is unchanged.

In [1]:
import sys, os, json, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not pathlib.Path("ERA_V5").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/amukka/ERA_V5.git"], check=True)
    os.chdir("ERA_V5/session13_Distributed_Training")
ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
if not (ROOT / "data" / "train.bin").exists():          # ~5 minutes: stream FineWeb-Edu, train the BPE
    subprocess.run([sys.executable, "tools/prepare_data.py"], check=True)

import torch, platform
from src import experiments as X
from src.train import pick_device
dev = pick_device()
print("device:", dev, "| torch", torch.__version__, "| python", platform.python_version(), "|", platform.platform())
if dev == "mps":
    print(f"MPS recommended working set: {torch.mps.recommended_max_memory()/2**30:.2f} GiB")
if dev == "cuda":
    print(torch.cuda.get_device_name(), f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
print(json.dumps(json.loads((ROOT / "data" / "meta.json").read_text()), indent=1))

device: mps | torch 2.13.0 | python 3.14.5 | macOS-26.6.2-arm64-arm-64bit-Mach-O
MPS recommended working set: 11.84 GiB
{
 "source": "HuggingFaceFW/fineweb-edu, config sample-10BT, streamed in order",
 "vocab_size": 8192,
 "eos_id": 0,
 "train_tokens": 56000000,
 "train_docs": 46080,
 "val_tokens": 1000000,
 "val_docs": 1024,
 "chars_per_token": 3.886,
 "note": "val.bin comes from the tokenizer's training documents; train.bin from the documents after them",
 "seconds": 314.3
}


**The model** (identical in every run): GPT, 10 layers, d_model 384, 6 heads, MLP 4x, sequence 512,
byte-level BPE vocabulary 8,192 with the head tied to the embedding. **21,053,184 parameters**, of which
17,710,848 are outside the embedding tables. No dropout anywhere, because a reversible backward pass
recomputes each block and a random mask would recompute differently. AdamW (0.9, 0.95, wd 0.1), peak LR
1e-3, 3% warmup then cosine to 10%, gradient clip 1.0, fp32.

In [2]:
chosen = json.loads((X.RESULTS / "chosen_variant.json").read_text())
print("searching the standard stack")
max_std, trail_std = X.max_batch("standard", 1.0, start=32, step=4)
print("searching the reversible stack:", chosen["variant"], "h =", chosen["h"])
max_rev, trail_rev = X.max_batch(chosen["variant"], chosen["h"], start=32, step=8)
print(f"\nmax batch  standard: {max_std}   {chosen['variant']}: {max_rev}   ratio {max_rev/max_std:.2f}x")
(X.RESULTS / "max_batch.json").write_text(json.dumps(
    {"standard": {"max_batch": max_std, "trail": trail_std},
     chosen["variant"]: {"max_batch": max_rev, "trail": trail_rev, "h": chosen["h"]}}, indent=2))

searching the standard stack


  standard  batch   32  peak    9,466 MiB   10,550 tok/s


  standard  batch   64  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 12.02 GiB, other a


  standard  batch   48  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 11.77 GiB, other a


  standard  batch   40  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 10.90 GiB, other a


...

  momentum  batch   96  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 10.03 GiB, other a


  momentum  batch   80  peak    6,754 MiB    7,860 tok/s


  momentum  batch   88  peak    8,157 MiB    7,745 tok/s



max batch  standard: 32   momentum: 88   ratio 2.75x


3332

In [3]:
import math
lr = 1e-3 * math.sqrt(max_rev / X.BASE_BATCH)
print(f"batch {max_rev} = {max_rev*512:,} tokens a step, {X.TOKENS // (max_rev*512):,} steps, peak LR {lr:.2e}")
big = X.train(name=f"reversible_{chosen['variant']}_maxbatch", variant=chosen["variant"], h=chosen["h"],
              batch=max_rev, tokens=X.TOKENS, lr=lr, warmup_frac=0.05, mem_fraction=1.0,
              eval_every=max(1, 250 * X.BASE_BATCH // max_rev), log_every=2)

batch 88 = 45,056 tokens a step, 1,109 steps, peak LR 1.66e-03


{"name": "reversible_momentum_maxbatch", "device": "mps", "config": {"vocab_size": 8192, "seq_len": 512, "n_layer": 10, "n_head": 6, "d_model": 384, "variant": "momentum", "h": 0.5, "fp_iters": 6, "extra": {}}, "params": 21053184, "params_non_embedding": 17710848, "batch": 88, "tokens_per_step": 45056, "steps": 1109, "peak_lr": 0.0016583123951777, "warmup": 55, "amp_bf16": 0, "torch": "2.13.0", "machine": "arm64", "processor": "arm"}


step     0/1109  tok   0.05M  loss 9.0855  gnorm 3.374  0 tok/s  0.1 min


memory audit at step 3: exact peak 6,354 MiB in aten._log_softmax_backward_data.default


step    20/1109  tok   0.90M  loss 7.3019  gnorm 0.592  7,692 tok/s  2.0 min


...

step  1080/1109  tok  48.66M  loss 4.0926  gnorm 0.535  8,369 tok/s  104.3 min


step  1100/1109  tok  49.56M  loss 4.0861  gnorm 0.487  7,733 tok/s  106.2 min


step  1108/1109  tok  49.92M  loss 4.0923  gnorm 0.495  7,800 tok/s  107.0 min


            eval @ 49.92M tokens: val loss 4.1808


reversibility at final weights: grad rel err 8.69e-05, cosine 1.000000, state rebuild err 6.50e-05
{"tok_per_s": 7776.4, "final_train_loss": 4.092309951782227, "final_val_loss": 4.180837237834931, "peak_exact_mib": 6354.4, "peak_alloc_mib": 3539.4, "peak_driver_mib": 11555.3, "train_seconds": 6419.7}


In [4]:
keys = ["batch", "tokens_trained", "steps", "peak_lr", "final_train_loss", "final_train_loss_avg_last5pct",
        "final_val_loss", "tok_per_s", "train_seconds", "peak_exact_mib", "peak_exact_op", "peak_alloc_mib", "peak_driver_mib"]
print(json.dumps({k: big[k] for k in keys}, indent=1))
print(json.dumps(big["reversibility"], indent=1))

{
 "batch": 88,
 "tokens_trained": 49922048,
 "steps": 1109,
 "peak_lr": 0.0016583123951777,
 "final_train_loss": 4.092309951782227,
 "final_train_loss_avg_last5pct": 4.162381719659876,
 "final_val_loss": 4.180837237834931,
 "tok_per_s": 7776.4,
 "train_seconds": 6419.7,
 "peak_exact_mib": 6354.4,
 "peak_exact_op": "aten._log_softmax_backward_data.default",
 "peak_alloc_mib": 3539.4,
 "peak_driver_mib": 11555.3
}
{
 "init": {
  "grad_rel_err": 2.200799507637422e-06,
  "grad_cosine": 0.9999999999972711,
  "state_rebuild_rel_err_per_layer": [
   5.324042831489351e-06,
   2.7300793590256944e-06,
   1.537154957986786e-06,
   9.912050700222608e-07,
   6.691512908219011e-07,
   5.152484163772897e-07,
   4.32310315545692e-07,
   2.8608965862986224e-07,
   2.3783076130712288e-07
  ],
  "state_rebuild_rel_err_max": 5.324042831489351e-06
 },
 "final": {
  "grad_rel_err": 8.689006283887884e-05,
  "grad_cosine": 0.9999999962297621,
  "state_rebuild_rel_err_per_layer": [
   6.500593735836446e-05,
 